# PICK

## 1. Cài đặt môi trường

In [ ]:
import sys
from pathlib import Path

# Cài đặt trên Kaggle
if Path("/kaggle").exists():
    if not Path("reconia").exists():
        !git clone -b sub-dev-02 -q https://github.com/tamir39/reconia.git
    %cd reconia
    !pip install -q -r requirements.txt

# Cấu hình đường dẫn import
repo_root = Path.cwd().resolve()
for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if path.name == "reconia":
        repo_root = path
        break
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))


## 2. Chuẩn bị dữ liệu

In [ ]:
from prepare.prepare_sroie import prepare_sroie
from prepare.prepare_cord_v2 import prepare_cord_v2
from prepare.prepare_wildreceipt import prepare_wildreceipt

prepare_sroie()
prepare_cord_v2()
prepare_wildreceipt()

## 3. Tải trọng số mô hình

In [ ]:
# PICK sử dụng kiến trúc đồ thị và trọng số riêng
# download_weights("pick")

## 4. Huấn luyện mô hình

In [ ]:
from models.pick.train import train_model

# Huấn luyện trên sroie
train_model(dataset="sroie", epochs=20, batch_size=4, accum_steps=4, lr=1e-4, patience=5)


In [ ]:
# Huấn luyện trên cord_v2
train_model(dataset="cord_v2", epochs=20, batch_size=4, accum_steps=4, lr=1e-4, patience=5)


In [ ]:
# Huấn luyện trên wildreceipt
train_model(dataset="wildreceipt", epochs=20, batch_size=4, accum_steps=4, lr=1e-4, patience=5)


## 5. Đánh giá kết quả

In [ ]:
import json
from pathlib import Path
from IPython.display import Markdown, display

rows = [
    "| Tập dữ liệu | Test Loss | Precision | Recall | F1 |",
    "| :--- | :---: | :---: | :---: | :---: |",
]
for dataset in ("sroie", "cord_v2", "wildreceipt"):
    path = repo_root / f"results/{dataset}/pick/test_metrics.json"
    if path.exists():
        m = json.loads(path.read_text(encoding="utf-8"))
        rows.append(f"| {dataset} | {m['test_loss']:.4f} | {m['precision']:.4f} | {m['recall']:.4f} | {m['f1']:.4f} |")

display(Markdown("\n".join(rows)))

## 6. Đóng gói kết quả (Tùy chọn)

In [ ]:
import shutil
from pathlib import Path
from IPython.display import FileLink, display

if Path("results").exists():
    shutil.make_archive("results", "zip", "results")
    display(FileLink("results.zip"))
